# Import

In [1]:
import numpy as np
from matplotlib import pyplot as plt
import qutip as qt
import scipy.sparse
import matplotlib as mpl
mpl.rcParams['figure.dpi']= 400
from matplotlib import rcParams
from scipy import constants as cst
import seaborn as sns
from scipy.sparse import *
from scipy.sparse.linalg import eigs
from scipy.linalg import expm, sinm, cosm
from numpy.linalg import matrix_power
from pyfonts import load_font
from scipy.sparse.linalg import eigsh
import scqubits as scq
import pandas as pd
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter

In [2]:
plt.style.use ('seaborn-v0_8-deep')
sns.set_style(style='white')
mpl.rcParams['figure.dpi']= 400
mpl.rc('lines',
       linewidth=1,
       dashed_pattern=[4, 3],
       dashdot_pattern=[4, 2, 1, 2],
       dotted_pattern=[1, 2])

mpl.rc('font',
       family='sans-serif',
       size=16)

mpl.rc('mathtext',
       fontset='cm')

mpl.rc('axes',
       linewidth=0.75,
       titlesize=16,
       labelsize=14,
       titlepad=10,
       )

mpl.rc('xtick',
       top=False,
       bottom = True,
       labelsize=12,
       direction='in')

mpl.rc('xtick.major',
       size=5,
       width=0.75)

mpl.rc('xtick.minor',
       size=2,
       width=0.75,
       visible=False)

plt.rc('ytick',
       right=False,
       left = True,
       labelsize=12,
       direction='in')

mpl.rc('ytick.major',
       size=5,
       width=0.75)

mpl.rc('ytick.minor',
       size=2,
       width=0.75,
       visible=False)

mpl.rc('legend',
       fontsize=12)

mpl.rc('figure',
       titlesize=12)

plt.rcParams["text.usetex"]=False
rcParams['font.serif'] = ['Gadugi']

# load font
font = load_font(
   font_url="https://github.com/fontfen/myriad-pro/blob/master/MyriadPro-Regular.ttf?raw=true"
)

# directory = r'C:\Users\nbaol\iCloudDrive\gridium_data\simulation\\'
# fig_directory = r'C:\Users\nbaol\iCloudDrive\gridium_paper\Illustration\\'

directory = r'/Users/longbnguyen/Library/Mobile Documents/com~apple~CloudDocs/gridium_data/simulation//'
fig_directory = r'/Users/longbnguyen/Library/Mobile Documents/com~apple~CloudDocs/gridium_paper/Illustration//'

fsize = [2.5,4]
lw = 0.75
lw2 = 1.5
labelsize = 14
fontsize = 14
phi_ext_array = np.linspace(0,1,101)

# Device A

## Symmetric

In [3]:
gkp_yaml = """branches:
- [JJ, 0,1, EJ=4.14, EC=0.77]
- [JJ, 0,2, EJ, EC]
- [C, 1,2, eK=1.67]
- [L, 1,4, EL=0.77]
- [L, 2,4, EL]
- [L, 3,4, EL1=0.77]
- [JJ, 0, 3, 4.14, 5.28]
- [C, 0,4, eC=2.24]
"""
gkp = scq.Circuit(gkp_yaml, from_file=False, ext_basis="discretized")

trans_mat = np.array([[1., 0., 0., 0.],
                      [1., 1., 0., 0.],
                      [1., 1., 1., 0.],
                      [1., 0., 0., 1.]])

gkp.configure(transformation_matrix=trans_mat, closure_branches=[gkp.branches[1], gkp.branches[-2]])
gkp.cutoff_n_1 = 10
gkp.cutoff_ext_2 = 150
gkp.cutoff_ext_3 = 150
gkp.cutoff_ext_4 = 100

gkp.configure(
    system_hierarchy=[[[1, 2], [4]], [3]],
    subsystem_trunc_dims=[[120, [100, 20]], 20],
)

In [4]:
gkp.Φ1 = 0.5
gkp.ng1 = 0

# ── Charge dispersion: f01(ng=0) vs f01(ng=0.5) ──────────────────────────────
gkp.Φ2 = 0.0
esys = gkp.eigensys()
w01_0 = esys[0][1] - esys[0][0]

gkp.ng1 = 0.5
esys = gkp.eigensys()
w01_x = esys[0][1] - esys[0][0]

# ── Flux dispersion: f01(Φ2=0) vs f01(Φ2=0.25), ng1=0 ───────────────────────
gkp.ng1 = 0
gkp.Φ2 = 0.0
f01_phi0 = gkp.eigensys()[0][1] - gkp.eigensys()[0][0]

gkp.Φ2 = 0.25
f01_phi25 = gkp.eigensys()[0][1] - gkp.eigensys()[0][0]

print("── Device A ───────────────────────────")
print(f"  f01 (ng=0):      {w01_0*1e3:.4f} MHz")
print(f"  f01 (ng=0.5):    {w01_x*1e3:.4f} MHz")
print(f"  Charge disp.:    {abs(w01_x - w01_0)*1e6:.2f} kHz")
print(f"  Flux disp.:      {abs(f01_phi25 - f01_phi0)*1e6:.2f} kHz  (Φ2=0 vs Φ2=0.25)")


── Device A ───────────────────────────
  f01 (ng=0):      43.2446 MHz
  f01 (ng=0.5):    19.6847 MHz
  Charge disp.:    23559.95 kHz
  Flux disp.:      792732.04 kHz  (Φ2=0 vs Φ2=0.25)


## Asymmetric

In [5]:
EJ = 4.14
eps = 0.1
EJ1 = EJ*(1+eps)
EJ2 = EJ*(1-eps)

gkp_yaml = f"""branches:
- [JJ, 0,1, {EJ1}, EC=0.76]
- [JJ, 0,2, {EJ2}, EC]
- [C, 1,2, eK=0.77]
- [L, 1,4, EL=0.77]
- [L, 2,4, EL]
- [L, 3,4, EL1=0.77]
- [JJ, 0, 3, 4.14, 4.79]
- [C, 0,4, eC=1.2]
"""
gkp = scq.Circuit(gkp_yaml, from_file=False, ext_basis="discretized")

trans_mat = np.array([[1., 0., 0., 0.],
                      [1., 1., 0., 0.],
                      [1., 1., 1., 0.],
                      [1., 0., 0., 1.]])

gkp.configure(transformation_matrix=trans_mat, closure_branches=[gkp.branches[1], gkp.branches[-2]])
gkp.cutoff_n_1 = 10
gkp.cutoff_ext_2 = 150
gkp.cutoff_ext_3 = 150
gkp.cutoff_ext_4 = 100

gkp.configure(
    system_hierarchy=[[[1, 2], [4]], [3]],
    subsystem_trunc_dims=[[120, [100, 20]], 20],
)

In [6]:
gkp.Φ1 = 0.5
gkp.ng1 = 0

# ── Charge dispersion: f01(ng=0) vs f01(ng=0.5) ──────────────────────────────
gkp.Φ2 = 0.0
esys = gkp.eigensys()
w01_0 = esys[0][1] - esys[0][0]

gkp.ng1 = 0.5
esys = gkp.eigensys()
w01_x = esys[0][1] - esys[0][0]

# ── Flux dispersion: f01(Φ2=0) vs f01(Φ2=0.25), ng1=0 ───────────────────────
gkp.ng1 = 0
gkp.Φ2 = 0.0
f01_phi0 = gkp.eigensys()[0][1] - gkp.eigensys()[0][0]

gkp.Φ2 = 0.25
f01_phi25 = gkp.eigensys()[0][1] - gkp.eigensys()[0][0]

print("── Device A ───────────────────────────")
print(f"  f01 (ng=0):      {w01_0*1e3:.4f} MHz")
print(f"  f01 (ng=0.5):    {w01_x*1e3:.4f} MHz")
print(f"  Charge disp.:    {abs(w01_x - w01_0)*1e6:.2f} kHz")
print(f"  Flux disp.:      {abs(f01_phi25 - f01_phi0)*1e6:.2f} kHz  (Φ2=0 vs Φ2=0.25)")

── Device A ───────────────────────────
  f01 (ng=0):      22.1237 MHz
  f01 (ng=0.5):    17.2605 MHz
  Charge disp.:    4863.20 kHz
  Flux disp.:      921818.48 kHz  (Φ2=0 vs Φ2=0.25)


# Device B

## Symmetric

In [7]:
gkp_yaml = """branches:
- [JJ, 0,1, EJ=7.21, EC=0.77]
- [JJ, 0,2, EJ, EC]
- [C, 1,2, eK=0.68]
- [L, 1,4, EL=0.73]
- [L, 2,4, EL]
- [L, 3,4, EL1=0.73]
- [JJ, 0, 3, 1.15, 8]
- [C, 0,4, eC=1]
"""
gkp = scq.Circuit(gkp_yaml, from_file=False, ext_basis = "discretized")
# set use_dynamic_flux_grouping=True to calculate the coherence times

trans_mat =  np.array([[1., 0., 0., 0.],
                       [1., 1., 0., 0.],
                       [1., 1., 1., 0.],
                       [1., 0., 0., 1.]])

gkp.configure(transformation_matrix=trans_mat, closure_branches=[gkp.branches[1], gkp.branches[-2]])
# include branch gkp.branches[-1] into closure_branches when using dynamic flux grouping
gkp.cutoff_n_1 = 10
gkp.cutoff_ext_2 = 150
gkp.cutoff_ext_3 = 150
gkp.cutoff_ext_4 = 100

gkp.configure(
              system_hierarchy=[[[1, 2], [4]], [3]], 
              subsystem_trunc_dims=[[120, [100, 20]], 20],
              )

In [8]:
gkp.Φ1 = 0.5
gkp.ng1 = 0

# ── Charge dispersion: f01(ng=0) vs f01(ng=0.5) ──────────────────────────────
gkp.Φ2 = 0.0
esys = gkp.eigensys()
w01_0 = esys[0][1] - esys[0][0]

gkp.ng1 = 0.5
esys = gkp.eigensys()
w01_x = esys[0][1] - esys[0][0]

# ── Flux dispersion: f01(Φ2=0) vs f01(Φ2=0.25), ng1=0 ───────────────────────
gkp.ng1 = 0
gkp.Φ2 = 0.0
f01_phi0 = gkp.eigensys()[0][1] - gkp.eigensys()[0][0]

gkp.Φ2 = 0.25
f01_phi25 = gkp.eigensys()[0][1] - gkp.eigensys()[0][0]

print("── Device B ───────────────────────────")
print(f"  f01 (ng=0):      {w01_0*1e3:.4f} MHz")
print(f"  f01 (ng=0.5):    {w01_x*1e3:.4f} MHz")
print(f"  Charge disp.:    {abs(w01_x - w01_0)*1e6:.2f} kHz")
print(f"  Flux disp.:      {abs(f01_phi25 - f01_phi0)*1e6:.2f} kHz  (Φ2=0 vs Φ2=0.25)")


── Device B ───────────────────────────
  f01 (ng=0):      0.6185 MHz
  f01 (ng=0.5):    0.0583 MHz
  Charge disp.:    560.26 kHz
  Flux disp.:      156221.23 kHz  (Φ2=0 vs Φ2=0.25)


## Asymmetric

In [9]:
EJ = 7.21
eps = 0.1
EJ1 = EJ*(1+eps)
EJ2 = EJ*(1-eps)

gkp_yaml = f"""branches:
- [JJ, 0,1, {EJ1}, EC=0.77]
- [JJ, 0,2, {EJ2}, EC]
- [C, 1,2, eK=0.68]
- [L, 1,4, EL=0.73]
- [L, 2,4, EL]
- [L, 3,4, EL1=0.73]
- [JJ, 0, 3, 1.15, 8]
- [C, 0,4, eC=1]
"""
gkp = scq.Circuit(gkp_yaml, from_file=False, ext_basis = "discretized")
# set use_dynamic_flux_grouping=True to calculate the coherence times

trans_mat =  np.array([[1., 0., 0., 0.],
                       [1., 1., 0., 0.],
                       [1., 1., 1., 0.],
                       [1., 0., 0., 1.]])

gkp.configure(transformation_matrix=trans_mat, closure_branches=[gkp.branches[1], gkp.branches[-2]])
# include branch gkp.branches[-1] into closure_branches when using dynamic flux grouping
gkp.cutoff_n_1 = 10
gkp.cutoff_ext_2 = 150
gkp.cutoff_ext_3 = 150
gkp.cutoff_ext_4 = 100

gkp.configure(
              system_hierarchy=[[[1, 2], [4]], [3]], 
              subsystem_trunc_dims=[[120, [100, 20]], 20],
              )

In [10]:
gkp.Φ1 = 0.5
gkp.ng1 = 0

# ── Charge dispersion: f01(ng=0) vs f01(ng=0.5) ──────────────────────────────
gkp.Φ2 = 0.0
esys = gkp.eigensys()
w01_0 = esys[0][1] - esys[0][0]

gkp.ng1 = 0.5
esys = gkp.eigensys()
w01_x = esys[0][1] - esys[0][0]

# ── Flux dispersion: f01(Φ2=0) vs f01(Φ2=0.25), ng1=0 ───────────────────────
gkp.ng1 = 0
gkp.Φ2 = 0.0
f01_phi0 = gkp.eigensys()[0][1] - gkp.eigensys()[0][0]

gkp.Φ2 = 0.25
f01_phi25 = gkp.eigensys()[0][1] - gkp.eigensys()[0][0]

print("── Device B ───────────────────────────")
print(f"  f01 (ng=0):      {w01_0*1e3:.4f} MHz")
print(f"  f01 (ng=0.5):    {w01_x*1e3:.4f} MHz")
print(f"  Charge disp.:    {abs(w01_x - w01_0)*1e6:.2f} kHz")
print(f"  Flux disp.:      {abs(f01_phi25 - f01_phi0)*1e6:.2f} kHz  (Φ2=0 vs Φ2=0.25)")

── Device B ───────────────────────────
  f01 (ng=0):      0.7723 MHz
  f01 (ng=0.5):    0.4489 MHz
  Charge disp.:    323.41 kHz
  Flux disp.:      155996.32 kHz  (Φ2=0 vs Φ2=0.25)


# Device C

## Symmetric

In [11]:
gkp_yaml = """branches:
- [JJ, 0,1, EJ=9, EC=0.77]
- [JJ, 0,2, 9, EC]
- [C, 1,2, eK=0.67]
- [L, 1,4, EL=0.5]
- [L, 2,4, EL]
- [L, 3,4, EL1=0.3]
- [JJ, 0, 3, 1.25, 8]
- [C, 0,4, eC=2.8]
"""
gkp = scq.Circuit(gkp_yaml, from_file=False, ext_basis="discretized")

trans_mat = np.array([[1., 0., 0., 0.],
                      [1., 1., 0., 0.],
                      [1., 1., 1., 0.],
                      [1., 0., 0., 1.]])

gkp.configure(transformation_matrix=trans_mat, closure_branches=[gkp.branches[1], gkp.branches[-2]])
gkp.cutoff_n_1 = 10
gkp.cutoff_ext_2 = 150
gkp.cutoff_ext_3 = 150
gkp.cutoff_ext_4 = 100

gkp.configure(
    system_hierarchy=[[[1, 2], [4]], [3]],
    subsystem_trunc_dims=[[120, [100, 20]], 20],
)

In [12]:
gkp.Φ1 = 0.5
gkp.ng1 = 0

# ── Charge dispersion: f01(ng=0) vs f01(ng=0.5) ──────────────────────────────
gkp.Φ2 = 0.0
esys = gkp.eigensys()
w01_0 = esys[0][1] - esys[0][0]

gkp.ng1 = 0.5
esys = gkp.eigensys()
w01_x = esys[0][1] - esys[0][0]

# ── Flux dispersion: f01(Φ2=0) vs f01(Φ2=0.25), ng1=0 ───────────────────────
gkp.ng1 = 0
gkp.Φ2 = 0.0
f01_phi0 = gkp.eigensys()[0][1] - gkp.eigensys()[0][0]

gkp.Φ2 = 0.25
f01_phi25 = gkp.eigensys()[0][1] - gkp.eigensys()[0][0]

print("── Device C ───────────────────────────")
print(f"  f01 (ng=0):      {w01_0*1e3:.4f} MHz")
print(f"  f01 (ng=0.5):    {w01_x*1e3:.4f} MHz")
print(f"  Charge disp.:    {abs(w01_x - w01_0)*1e6:.2f} kHz")
print(f"  Flux disp.:      {abs(f01_phi25 - f01_phi0)*1e6:.2f} kHz  (Φ2=0 vs Φ2=0.25)")


── Device C ───────────────────────────
  f01 (ng=0):      0.1396 MHz
  f01 (ng=0.5):    0.0029 MHz
  Charge disp.:    136.67 kHz
  Flux disp.:      42148.27 kHz  (Φ2=0 vs Φ2=0.25)


## Asymmetric

In [13]:
EJ = 9
eps = 0.1
EJ1 = EJ*(1+eps)
EJ2 = EJ*(1-eps)

gkp_yaml = f"""branches:
- [JJ, 0,1, {EJ1}, EC=0.77]
- [JJ, 0,2, {EJ2}, EC]
- [C, 1,2, eK=0.67]
- [L, 1,4, EL=0.5]
- [L, 2,4, EL]
- [L, 3,4, EL1=0.3]
- [JJ, 0, 3, 1.25, 8]
- [C, 0,4, eC=2.8]
"""
gkp = scq.Circuit(gkp_yaml, from_file=False, ext_basis="discretized")

trans_mat = np.array([[1., 0., 0., 0.],
                      [1., 1., 0., 0.],
                      [1., 1., 1., 0.],
                      [1., 0., 0., 1.]])

gkp.configure(transformation_matrix=trans_mat, closure_branches=[gkp.branches[1], gkp.branches[-2]])
gkp.cutoff_n_1 = 10
gkp.cutoff_ext_2 = 150
gkp.cutoff_ext_3 = 150
gkp.cutoff_ext_4 = 100

gkp.configure(
    system_hierarchy=[[[1, 2], [4]], [3]],
    subsystem_trunc_dims=[[120, [100, 20]], 20],
)

In [14]:
gkp.Φ1 = 0.5
gkp.ng1 = 0

# ── Charge dispersion: f01(ng=0) vs f01(ng=0.5) ──────────────────────────────
gkp.Φ2 = 0.0
esys = gkp.eigensys()
w01_0 = esys[0][1] - esys[0][0]

gkp.ng1 = 0.5
esys = gkp.eigensys()
w01_x = esys[0][1] - esys[0][0]

# ── Flux dispersion: f01(Φ2=0) vs f01(Φ2=0.25), ng1=0 ───────────────────────
gkp.ng1 = 0
gkp.Φ2 = 0.0
f01_phi0 = gkp.eigensys()[0][1] - gkp.eigensys()[0][0]

gkp.Φ2 = 0.25
f01_phi25 = gkp.eigensys()[0][1] - gkp.eigensys()[0][0]

print("── Device C ───────────────────────────")
print(f"  f01 (ng=0):      {w01_0*1e3:.4f} MHz")
print(f"  f01 (ng=0.5):    {w01_x*1e3:.4f} MHz")
print(f"  Charge disp.:    {abs(w01_x - w01_0)*1e6:.2f} kHz")
print(f"  Flux disp.:      {abs(f01_phi25 - f01_phi0)*1e6:.2f} kHz  (Φ2=0 vs Φ2=0.25)")


── Device C ───────────────────────────
  f01 (ng=0):      0.1685 MHz
  f01 (ng=0.5):    0.0903 MHz
  Charge disp.:    78.18 kHz
  Flux disp.:      42108.64 kHz  (Φ2=0 vs Φ2=0.25)
